# 01 — Auditoría de calidad de datos (A1)
Objetivo: demostrar que los datos de RetailVenture son confiables antes de analizarlos.
El notebook tiene tres bloques:
1. Perfil general: ¿cada tabla está sana por sí sola?
2. Integridad y reglas de negocio: ¿las tablas son coherentes entre sí y con el negocio?
3. Métricas de calidad y exportación de tablas limpias.

## Bloque 1 — Perfil general
Para cada tabla: filas, columnas, nulos, filas duplicadas, claves primarias duplicadas
y rango de fechas.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")
CLEAN.mkdir(parents=True, exist_ok=True)

FECHAS = {
    "clientes": ["fecha_registro"],
    "transacciones": ["fecha"],
    "productos": [],
    "devoluciones": ["fecha_devolucion"],
    "interacciones_crm": ["fecha"],
    "campanas_marketing": ["fecha_envio"],
}
PK = {
    "clientes": "cliente_id",
    "transacciones": "transaccion_id",
    "productos": "producto_id",
    "devoluciones": "devolucion_id",
    "interacciones_crm": "interaccion_id",
    "campanas_marketing": "campana_id",
}

dfs = {n: pd.read_csv(RAW / f"{n}.csv", parse_dates=f) for n, f in FECHAS.items()}

perfil = pd.DataFrame([
    {
        "tabla": n,
        "filas": len(df),
        "columnas": df.shape[1],
        "nulos": int(df.isna().sum().sum()),
        "filas_duplicadas": int(df.duplicated().sum()),
        "pk_duplicadas": int(df[PK[n]].duplicated().sum()),
        "desde": df[FECHAS[n][0]].min().date() if FECHAS[n] else None,
        "hasta": df[FECHAS[n][0]].max().date() if FECHAS[n] else None,
    }
    for n, df in dfs.items()
])
perfil

,tabla,filas,columnas,nulos,filas_duplicadas,pk_duplicadas,desde,hasta
0,clientes,8000,7,0,0,0,2023-01-01,2024-12-31
1,transacciones,50000,10,0,0,0,2023-01-01,2024-12-31
2,productos,318,7,0,0,0,None,None
3,devoluciones,6724,6,0,0,0,2023-01-02,2024-12-31
4,interacciones_crm,4684,7,0,0,0,2023-01-01,2024-12-31
5,campanas_marketing,8820,8,0,0,0,2023-01-01,2024-12-31


**Hallazgo:** cero nulos y cero duplicados en las 6 tablas; los datos cubren
enero 2023 – diciembre 2024. Productos tiene 318 registros, no 320 como dice
el enunciado.

## Bloque 2 — Integridad referencial y reglas de negocio
Pregunta: ¿las tablas son coherentes entre sí y con la lógica del negocio?
Primero revisamos que cada clave foránea apunte a un registro existente; después,
reglas que los datos deberían cumplir. Cada regla cuenta las filas que la violan:
0 significa que se cumple.

In [2]:
cli, tx, pr, dev, crm, cam = (dfs[n] for n in FECHAS)

relaciones = [
    ("transacciones.cliente_id", tx.cliente_id, cli.cliente_id),
    ("transacciones.producto_id", tx.producto_id, pr.producto_id),
    ("devoluciones.transaccion_id", dev.transaccion_id, tx.transaccion_id),
    ("devoluciones.cliente_id", dev.cliente_id, cli.cliente_id),
    ("interacciones_crm.cliente_id", crm.cliente_id, cli.cliente_id),
    ("campanas_marketing.cliente_id", cam.cliente_id, cli.cliente_id),
]
integridad = pd.DataFrame([
    {"relacion": r, "huerfanos": int((~hijo.isin(padre)).sum()), "total": len(hijo)}
    for r, hijo, padre in relaciones
])
integridad

,relacion,huerfanos,total
0,transacciones.cliente_id,0,50000
1,transacciones.producto_id,0,50000
2,devoluciones.transaccion_id,0,6724
3,devoluciones.cliente_id,0,6724
4,interacciones_crm.cliente_id,0,4684
5,campanas_marketing.cliente_id,0,8820


In [3]:
dv = dev.merge(tx, on="transaccion_id", suffixes=("_dev", "_tx"))
txp = tx.merge(pr, on="producto_id")
tc = tx.merge(cli[["cliente_id", "fecha_registro"]], on="cliente_id")
dias_dev = (dv.fecha_devolucion - dv.fecha).dt.days
antes_registro = tc.fecha < tc.fecha_registro
margen_calc = (pr.precio_lista - pr.costo) / pr.precio_lista * 100

reglas = {
    "Devolución de otro cliente distinto al de la compra": (dv.cliente_id_dev != dv.cliente_id_tx).sum(),
    "Transacción con más de una devolución": (dev.transaccion_id.value_counts() > 1).sum(),
    "monto_neto ≠ cantidad × precio × (1 − descuento)": (abs(tx.cantidad * tx.precio_unitario * (1 - tx.descuento) - tx.monto_neto) > 0.01).sum(),
    "Cantidad ≤ 0": (tx.cantidad <= 0).sum(),
    "Descuento fuera de [0, 1)": ((tx.descuento < 0) | (tx.descuento >= 1)).sum(),
    "Precio unitario ≠ precio de lista": (abs(txp.precio_unitario - txp.precio_lista) > 0.01).sum(),
    "Precio unitario < costo": (txp.precio_unitario < txp.costo).sum(),
    "margen_bruto_pct no cuadra con costo y precio": (abs(margen_calc - pr.margen_bruto_pct) > 0.5).sum(),
    "Devolución antes de la compra": (dias_dev < 0).sum(),
    "Devolución más de 30 días después": (dias_dev > 30).sum(),
    "Monto devuelto > monto de la compra": (dv.monto_devuelto > dv.monto_neto + 0.01).sum(),
    "Edad fuera de [18, 90]": ((cli.edad < 18) | (cli.edad > 90)).sum(),
    "Tiempo de respuesta CRM < 0": (crm.tiempo_respuesta_hrs < 0).sum(),
    "Conversión sin apertura": ((cam.convertido == 1) & (cam.abierto == 0)).sum(),
    "Transacciones antes del registro del cliente": antes_registro.sum(),
    "Clientes con compras antes de su registro": tc[antes_registro].cliente_id.nunique(),
    "Clientes que nunca compraron": (~cli.cliente_id.isin(tx.cliente_id)).sum(),
    "Tiempo de respuesta CRM > 100 h (outlier)": (crm.tiempo_respuesta_hrs > 100).sum(),
}
reglas_df = pd.DataFrame({"regla": reglas.keys(), "casos": [int(v) for v in reglas.values()]})
reglas_df

,regla,casos
0,Devolución de otro cliente distinto al de la c...,0
1,Transacción con más de una devolución,0
2,monto_neto ≠ cantidad × precio × (1 − descuento),0
3,Cantidad ≤ 0,0
4,"Descuento fuera de [0, 1)",0
5,Precio unitario ≠ precio de lista,0
6,Precio unitario < costo,0
7,margen_bruto_pct no cuadra con costo y precio,0
8,Devolución antes de la compra,0
9,Devolución más de 30 días después,0


In [4]:
ventas_inactivo = tx.merge(cli, on="cliente_id").query("segmento == 'Inactivo'").monto_neto.sum()
print(f"Transacciones antes del registro: {antes_registro.mean():.1%} del total")
print(f"Clientes compradores afectados: {tc[antes_registro].cliente_id.nunique() / tx.cliente_id.nunique():.1%}")
print(f"Ventas del segmento 'Inactivo': ${ventas_inactivo:,.0f}")

Transacciones antes del registro: 49.9% del total
Clientes compradores afectados: 76.3%
Ventas del segmento 'Inactivo': $317,287


**Hallazgos:**
- Integridad referencial perfecta: cero huérfanos en las 6 relaciones.
- Las reglas aritméticas y de rango se cumplen todas (monto neto, márgenes, plazo de devolución de 0 a 30 días).
- Problemas de lógica de negocio:
  1. **El 50% de las transacciones ocurre antes de la fecha de registro del cliente**, y afecta al 76% de los compradores. Tratamiento (decisión D6): conservar `fecha_registro` y crear `fecha_alta_efectiva` = primera compra.
  2. **392 clientes nunca compraron**: se conservan, pero se excluyen del CLV y del churn.
  3. **66 tiempos de respuesta del CRM por encima de 100 h** (máximo 518 h): se marcan como outliers y no se eliminan.
  4. **El segmento "Inactivo" vende $317K**: el segmento es estático y no refleja el comportamiento actual; se recalculará con RFM en A2.
  5. `convertido` en campañas no se vincula a ninguna transacción: el vínculo se reconstruirá en A4 por cliente y ventana de días.

## Bloque 3 — Métricas de calidad y tablas limpias
Resumimos la auditoría en un porcentaje por dimensión de calidad y generamos en
data/clean las tablas con las correcciones aplicadas. Esas tablas son el punto de
partida de los notebooks 02 a 04. data/raw nunca se modifica.

In [5]:
total_celdas = sum(df.size for df in dfs.values())
total_nulos = sum(int(df.isna().sum().sum()) for df in dfs.values())
total_filas = sum(len(df) for df in dfs.values())
total_pk_dup = sum(int(df[PK[n]].duplicated().sum()) for n, df in dfs.items())
total_fk = int(integridad.total.sum())
total_huerfanos = int(integridad.huerfanos.sum())
crm_outliers = int((crm.tiempo_respuesta_hrs > 100).sum())
tx_antes_registro = int(antes_registro.sum())

metricas = pd.DataFrame([
    ["Completitud", "Celdas no nulas", total_celdas - total_nulos, total_celdas],
    ["Unicidad", "Filas con clave primaria única", total_filas - total_pk_dup, total_filas],
    ["Integridad referencial", "Claves foráneas con registro padre", total_fk - total_huerfanos, total_fk],
    ["Validez", "Tiempos de respuesta CRM ≤ 100 h", len(crm) - crm_outliers, len(crm)],
    ["Consistencia temporal", "Transacciones posteriores al registro del cliente", len(tx) - tx_antes_registro, len(tx)],
], columns=["dimension", "regla", "validos", "total"])
metricas["pct_calidad"] = (metricas.validos / metricas.total * 100).round(1)
metricas

,dimension,regla,validos,total,pct_calidad
0,Completitud,Celdas no nulas,701918,701918,100.0
1,Unicidad,Filas con clave primaria única,78546,78546,100.0
2,Integridad referencial,Claves foráneas con registro padre,126952,126952,100.0
3,Validez,Tiempos de respuesta CRM ≤ 100 h,4618,4684,98.6
4,Consistencia temporal,Transacciones posteriores al registro del cliente,25028,50000,50.1


### Tratamiento aplicado (decisión D6 y hallazgos del bloque 2)
- **clientes**: se conserva `fecha_registro` y se agregan `primera_compra`, `ultima_compra`,
  `n_compras`, `fecha_alta_efectiva` y dos marcas:
  - `es_lead` = True si el cliente nunca compró (392). Un lead no es cliente hasta que compra.
  - `registro_posterior_a_compra` = True si compró antes de su fecha de registro.
  - `fecha_alta_efectiva` = primera compra; para los leads, la fecha de registro.
    **Para cohortes, antigüedad, CLV y churn, filtrar `es_lead == False`.**
- **transacciones**: se agrega la marca `antes_del_registro`.
- **interacciones_crm**: se agrega la marca `outlier_respuesta` (> 100 h). No se elimina ninguna fila.
- **productos, devoluciones, campanas_marketing**: se copian sin cambios.

In [6]:
resumen_compras = (tx.groupby("cliente_id").fecha
                   .agg(primera_compra="min", ultima_compra="max", n_compras="count")
                   .reset_index())
clientes_clean = cli.merge(resumen_compras, on="cliente_id", how="left")
clientes_clean["n_compras"] = clientes_clean.n_compras.fillna(0).astype(int)
clientes_clean["es_lead"] = clientes_clean.n_compras == 0
clientes_clean["fecha_alta_efectiva"] = clientes_clean.primera_compra.fillna(clientes_clean.fecha_registro)
clientes_clean["registro_posterior_a_compra"] = clientes_clean.primera_compra < clientes_clean.fecha_registro

transacciones_clean = tx.copy()
transacciones_clean["antes_del_registro"] = antes_registro.values

crm_clean = crm.copy()
crm_clean["outlier_respuesta"] = crm_clean.tiempo_respuesta_hrs > 100

limpias = {
    "clientes": clientes_clean,
    "transacciones": transacciones_clean,
    "productos": pr,
    "devoluciones": dev,
    "interacciones_crm": crm_clean,
    "campanas_marketing": cam,
}
for nombre, df in limpias.items():
    df.to_csv(CLEAN / f"{nombre}.csv", index=False)

pd.DataFrame({
    "tabla": limpias.keys(),
    "filas": [len(df) for df in limpias.values()],
    "columnas_nuevas": [len(df.columns) - len(dfs[n].columns) for n, df in limpias.items()],
})

,tabla,filas,columnas_nuevas
0,clientes,8000,6
1,transacciones,50000,1
2,productos,318,0
3,devoluciones,6724,0
4,interacciones_crm,4684,1
5,campanas_marketing,8820,0


In [7]:
# Verificaciones: ninguna fila se perdió y las marcas cuadran con el bloque 2
assert all(len(limpias[n]) == len(dfs[n]) for n in dfs)
assert clientes_clean.es_lead.sum() == 392
assert clientes_clean.registro_posterior_a_compra.sum() == tc[antes_registro].cliente_id.nunique()
assert transacciones_clean.antes_del_registro.sum() == tx_antes_registro
assert crm_clean.outlier_respuesta.sum() == crm_outliers
print("Verificaciones OK")

Verificaciones OK


**Conclusión de la auditoría A1:** los datos son técnicamente impecables (completitud,
unicidad e integridad al 100%), pero tienen un problema serio de lógica de negocio: solo el
50,1% de las transacciones es temporalmente consistente con el registro del cliente. No se
eliminó ninguna fila; las anomalías quedaron marcadas para que cada análisis decida cómo
tratarlas, y la decisión queda documentada en D6.